In [46]:
import numpy as np
import re

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import CXGate,  PauliEvolutionGate, QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp


from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router_rzz import CommutingGateRouterRzz
from qiskit_qaoa.utils.commuting_gate_router import CommutingGateRouter
from qiskit_qaoa.utils.commuting_gate_router_all_to_all import CommutingGateRouterAllToAll

from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph


In [47]:
def print_circuit_info(qc: QuantumCircuit, circuit_name):
    print(f'{circuit_name} has {qc.num_qubits} qubits')
    print(f'{circuit_name} has {qc.num_nonlocal_gates()} non-local gates and {qc.depth(lambda instr: len(instr.qubits) > 1)} non-local depth')
    print(f'{circuit_name} contains {list(qc.count_ops().keys())} gates.')

In [48]:
filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/test_N2_W2.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, [1,1])
hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
# ess = ExtendedSwapStrategy.from_grid(n, T)
ess = ExtendedSwapStrategy.from_all_to_all(n * T)
num_physical_qubits = ess._num_vertices
max_layers = 0

Keeping constraints at times: [0]


In [49]:
method = 'statevector'
backend_options = dict(
    method=method,
    device='CPU',
    precision='single',
    basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx"]
)

config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_physical_qubits
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, coupling_map=ess._coupling_map, **backend_options)
backend.set_option("n_qubits", num_physical_qubits)
print(f'Num qubits in backend: {backend.configuration().to_dict()["n_qubits"]}')

Num qubits in backend: 6


In [50]:
# default = QAOAAnsatz(hamiltonian, reps=1, initial_state=QuantumCircuit(num_physical_qubits), mixer_operator=QuantumCircuit(num_physical_qubits))
# t_default = transpile(default, backend, optimization_level=3,basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx"])
# t_default.draw(fold=-1)

In [51]:
# print_circuit_info(t_default, 'Default QAOA')

In [52]:
pm = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouterRzz(
            ess,
            max_layers=max_layers,
            perform_extra_swaps=True
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [53]:
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
tqc = pm.run(qc)

print_circuit_info(tqc, 'Rzz circuit')
print(tqc.count_ops())


swaps = []
for instruction in tqc.data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            swaps.append((int(matches[0]), int(matches[1])))
        else:
            raise Exception('Did not find 2 swap indices')
for swap in swaps[::-1]:
    tqc.swap(*swap)

# tqc.save_unitary()

Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rzz circuit has 6 qubits
Rzz circuit has 75 non-local gates and 62 non-local depth
Rzz circuit contains ['cx', 'rzz', 'rz'] gates.
OrderedDict([('cx', 46), ('rzz', 29), ('rz', 2)])


In [54]:
# tqc.draw(fold=-1)

Rotation site (1,3)
Gates (1,2,3), (1,2,3,4,5), (0,1,2,3,4,5), (0,1,3,5), (1,3,5)

In [55]:
# res_rzz = backend.run(tqc).result()
# data = res_rzz.results[0].data
# unitary = np.asarray(data.unitary)

In [56]:
pm_rz = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouter(
            ess,
            max_layers=max_layers,
            perform_extra_swaps=True
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [57]:
tqc_rz = pm_rz.run(qc)
print_circuit_info(tqc_rz, 'Rz circuit')
print(tqc_rz.count_ops())

swaps = []
for instruction in tqc_rz.data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            swaps.append((int(matches[0]), int(matches[1])))
        else:
            raise Exception('Did not find 2 swap indices')
for swap in swaps[::-1]:
    tqc_rz.swap(*swap)
# tqc_rz.save_unitary()
# res_rz = backend.run(tqc_rz).result()
# data_rz = res_rz.results[0].data
# unitary_rz = np.asarray(data_rz.unitary)

Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 6 qubits
Rz circuit has 52 non-local gates and 40 non-local depth
Rz circuit contains ['cx', 'rz'] gates.
OrderedDict([('cx', 52), ('rz', 31)])


In [58]:
tqc_rz.draw(fold=-1)

global phase: 0.34569
         ┌───────────┐                                                                                                                                                                                                  ┌───┐┌────────────┐                         ┌───┐┌───────────┐ ┌───┐┌───────────┐┌───┐             ┌───┐┌────────────┐             ┌───┐                                                                                                                                                                                                                                                     
q_0 -> 0 ┤ Rz(8.875) ├────────────────■─────────────────────────────────────────────────────■───────────────────────────────────■─────────────────────────────────────────────────────■─────────────────────────────────┤ X ├┤ Rz(-0.125) ├─────────────────────────┤ X ├┤ Rz(1.125) ├─┤ X ├┤ Rz(1.125) ├┤ X ├─────────────┤ X ├┤ Rz(-0.625) ├─────────────┤ X ├────────────────────────────────────■─────────────────────────────────────■───────────────────────────────────────────────■─────────────────────────────────────■────────────────────────────────────────────────────■──────────────────■────────────
         └───┬───┬───┘ ┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐└─┬─┘└───┬───┬────┘          ┌───┐┌───┐┌───┐└─┬─┘└───────────┘ └─┬─┘└───────────┘└─┬─┘┌───────────┐└─┬─┘└───┬───┬────┘┌───────────┐└─┬─┘┌───┐     ┌───┐                     │                                     │                     ┌───┐┌───────────┐ ┌───┐  │                                     │                                                    │                  │            
q_1 -> 1 ────┤ X ├─────┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├──┼──────┤ X ├───────────────┤ X ├┤ X ├┤ X ├──■──────────────────┼─────────────────■──┤ Rz(1.125) ├──┼──────┤ X ├─────┤ Rz(1.125) ├──┼──┤ X ├─────┤ X ├─────────────────────┼─────────────────────────────────────┼─────────────────────┤ X ├┤ Rz(0.625) ├─┤ X ├──┼─────────────────────────────────────┼────────────────────────────────────────────────────┼──────────────────┼────────────
             └─┬─┘     └───────────┘└───┘└───────────┘└─┬─┘└───────────┘└─┬─┘└───────────┘├───┤└───────────┘└─┬─┘└───────────┘└───┘└───────────┘└─┬─┘└───────────┘└─┬─┘└───────────┘└───┘└───────────┘└─┬─┘└───────────┘  │      └─┬─┘     ┌───┐     └─┬─┘└─┬─┘└─┬─┘┌───┐┌────────────┐  │                    └───────────┘  │      └─┬─┘     └───────────┘  │  └─┬─┘┌───┐└─┬─┘┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐└─┬─┘└───────────┘ └─┬─┘  │                                     │                     ┌───┐     ┌───┐┌────────────┐┌─┴─┐┌────────────┐┌─┴─┐┌───┐     
q_2 -> 2 ──────■────────────────────────────────────────┼─────────────────┼───────────────┤ X ├───────────────┼───────────────────────────────────■─────────────────┼───────────────────────────────────┼─────────────────┼────────■───────┤ X ├───────┼────■────┼──┤ X ├┤ Rz(-0.625) ├──┼───────────────────────────────────■────────┼──────────────────────■────┼──┤ X ├──┼──┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├──■──────────────────■────┼─────────────────────────────────────┼─────────────────────┤ X ├─────┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ X ├─────
         ┌────────────┐                                 │                 │               └─┬─┘               │                                                     │                                   │                 │                └─┬─┘       │         │  └─┬─┘└────────────┘  │                                            │                           │  └─┬─┘  │  └─┬─┘└────────────┘└───

In [42]:
# np.nonzero(np.abs(unitary - unitary_rz) > 1e-5)

In [43]:
# pm_rz_a2a = PassManager(
#     [
#         FindCommutingPauliEvolutionsMulti(), 
#         CommutingGateRouterAllToAll(
#             ess,
#         ),
#         SwapToFinalMapping(),
#         InverseCancellation(gates_to_cancel=[CXGate()]),
#         CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
#         InverseCancellation(gates_to_cancel=[CXGate()]),
#     ]
# )

In [44]:
# tqc_rz_a2a = pm_rz_a2a.run(qc)
# print_circuit_info(tqc_rz_a2a, 'Rz circuit a2a')
# print(tqc_rz_a2a.count_ops())


# swaps = []
# for instruction in tqc_rz_a2a.data:
#     if instruction.operation.name == 'swap':
#         qubits_str = str(instruction.qubits)
#         matches = re.findall('index=([0-9]+)', qubits_str)
#         if len(matches) == 2:
#             swaps.append((int(matches[0]), int(matches[1])))
#         else:
#             raise Exception('Did not find 2 swap indices')
# for swap in swaps[::-1]:
#     tqc_rz_a2a.swap(*swap)
# # tqc_rz_a2a.save_unitary()
# # res_rz_a2a = backend.run(tqc_rz_a2a).result()
# # data_rz_a2a = res_rz_a2a.results[0].data
# # unitary_rz_a2a = np.asarray(data_rz_a2a.unitary)

In [45]:
# tqc_rz_a2a.draw(fold=-1)